# flypair — connectome-simulated fruit flies in closed loop (Colab)

Runs top-to-bottom on a **fresh free-tier runtime**. Use **Runtime ▸ Change runtime type ▸ T4 GPU** for ~10× faster brains (CPU works too, just slower).

Each cell is numbered as in the project spec. Cells 1–5 take ~10 min the first time (1.1 GB MaleCNS download + build); afterwards everything is cached (optionally on Drive).

## 1. Get the code + install

In [ ]:
#@title 1. Clone repo + pip install  { display-mode: "form" }
REPO_URL = "https://github.com/YOUR_USER/flypair"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
import os, subprocess, sys
if os.path.exists("/content/flypair"):
    print("repo already present")
else:
    r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "/content/flypair"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr)
        print("Clone failed. Either fix REPO_URL above, or upload a zip of the repo:")
        from google.colab import files
        up = files.upload()  # pick flypair.zip
        name = next(iter(up))
        subprocess.run(["unzip", "-q", "-o", name, "-d", "/content"], check=True)
        if not os.path.exists("/content/flypair"):
            top = [d for d in os.listdir("/content") if os.path.isdir(f"/content/{d}") and os.path.exists(f"/content/{d}/pyproject.toml")]
            os.rename(f"/content/{top[0]}", "/content/flypair")
%cd /content/flypair
!pip install -q -e . 2>&1 | tail -1
!apt-get -qq install -y ffmpeg > /dev/null 2>&1 || true
import torch, flypair, psutil
print("flypair", flypair.__version__, "| torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "", "| RAM total", round(psutil.virtual_memory().total/2**30,1), "GB")

## 2. Optional: mount Google Drive for the connectome cache
Skip this cell to keep the cache in the runtime (rebuilt every new session, ~5–8 min).

In [ ]:
#@title 2. (optional) Drive cache  { display-mode: "form" }
USE_DRIVE = True  #@param {type:"boolean"}
CACHE_DIR = "/content/cache"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = "/content/drive/MyDrive/flypair_cache"
import os; os.makedirs(CACHE_DIR, exist_ok=True)
print("cache dir:", CACHE_DIR)

## 3. Download + build connectome(s)
MaleCNS v1.0: three feather files (1.1 GB) from the Janelia FlyEM bucket, streamed in batches (peak RAM ≈ 3 GB). FlyWire v783: the 100 MB parquet the Shiu repo ships + the Schlegel et al. annotation TSV. Both cached as `W.npz + neurons.parquet`.

In [ ]:
#@title 3. Build connectomes (RAM + timing printed)  { display-mode: "form" }
BUILD_MALECNS = True   #@param {type:"boolean"}
BUILD_FLYWIRE = True   #@param {type:"boolean"}
import time, psutil, gc
from flypair.connectome import load_connectome
conns = {}
for name, flag in (("malecns", BUILD_MALECNS), ("flywire", BUILD_FLYWIRE)):
    if not flag: continue
    t0 = time.time(); rss0 = psutil.Process().memory_info().rss/2**30
    conns[name] = load_connectome(name, cache_dir=CACHE_DIR)
    gc.collect()
    print(f"  -> {name}: {time.time()-t0:.0f} s, RSS now {psutil.Process().memory_info().rss/2**30:.2f} GB (was {rss0:.2f}); free RAM {psutil.virtual_memory().available/2**30:.1f} GB")
for c in conns.values(): print(c.summary())

## 4. Group-resolution report
Every semantic group → annotation regex → neuron count. Groups that do not exist in a dataset are listed as `UNRES` with the reason (e.g. Gr32a is not annotated anywhere; vpoDN/pIP10 are sex-specific).

In [ ]:
#@title 4. Group report
from flypair.groups import resolve_for
registries = {n: resolve_for(c, strict=False) for n, c in conns.items()}

## 5. Single-fly sanity check: sugar GRNs → MN9 (Shiu et al. 2024)
On **FlyWire** this drives Shiu's own 21 sugar-GRN IDs at 100 Hz and should give MN9 ≈ 67 Hz (their shipped result; ≈ 93 Hz at 150 Hz). On **MaleCNS** the same constants are reused unchanged (no published reference; mlx reports MN9_L ≈ 53 Hz). Also benchmarks CPU vs GPU and picks the faster device for the rest of the notebook.

In [ ]:
#@title 5. Sanity check + device benchmark
from flypair.sanity import sugar_mn9, benchmark_devices
first = next(iter(conns.values()))
bench = benchmark_devices(first, n_flies=2, n_steps=300)
DEVICE = bench["best"]
for name, c in conns.items():
    sugar_mn9(c, rate_hz=100.0, duration_ms=1000, n_trials=3, device=DEVICE)

## 6. Pick a scenario and run it
Scenarios live in `scenarios/*.yaml` (see `custom_template.yaml` for every option). `male_female` needs both connectomes.

In [ ]:
#@title 6. Run scenario  { display-mode: "form" }
SCENARIO = "male_male_intact"  #@param ["male_male_intact", "male_male_brake_removed", "courtship_chain", "male_female", "clone_mirror", "custom_template"]
DURATION_MS = 2000  #@param {type:"number"}
OUT_DIR = "/content/runs"
from flypair.scenario import load_scenario, run_scenario
scn = load_scenario(SCENARIO)
scn["duration_ms"] = DURATION_MS
need = {f["connectome"] for f in scn["flies"] if f.get("connectome")}
missing = need - set(conns)
assert not missing, f"scenario needs {missing}: enable them in cell 3"
live = run_scenario(scn, conns, out_dir=OUT_DIR, device=DEVICE)
print(f"ETA for controls: ~{live.meta['wall_s']:.0f} s each")

## 7. Controls: open-loop, playback, shuffled connectome + coupling metric
`playback`: fly A is replayed from the live recording (no brain), B runs live → tests whether the coupling is really bidirectional. `shuffled`: degree-preserving random rewiring. `open_loop`: all channels off.

In [ ]:
#@title 7. Controls + coupling metric  { display-mode: "form" }
RUN_CONTROLS = ["open_loop", "playback", "shuffled"]  #@param {type:"raw"}
from flypair.controls import run_controls
from flypair.metrics import coupling_report
runs = run_controls(scn, conns, live, controls=RUN_CONTROLS, out_dir=OUT_DIR, device=DEVICE)
A, B = scn["flies"][0]["name"], scn["flies"][1]["name"]
print(coupling_report(runs, A, B, groups=("p1", "pip10", "dna02", "dnp01")))

## 8. Figures + MP4

In [ ]:
#@title 8. Rasters, rates, trajectories, arena MP4
from pathlib import Path
from IPython.display import display, HTML, Image
import base64
from flypair.plots import rasters, rates, trajectories
from flypair.video import render
d = Path(OUT_DIR) / live.name
for fn, name in ((rasters, "rasters"), (rates, "rates"), (trajectories, "trajectories")):
    try:
        fig = fn(live, out=d / f"{name}.png"); display(fig)
    except ValueError as e:
        print(name, ":", e)
mp4 = render(live, d / "arena.mp4", fps=25, speed=1.0)
print("video:", mp4)
b64 = base64.b64encode(open(mp4, "rb").read()).decode()
display(HTML(f'<video width="900" controls src="data:video/mp4;base64,{b64}"></video>'))
from google.colab import files
files.download(str(mp4))

## 8b. 3D video (procedural fly model, three.js, headless Chromium)
Renders `viewer/index.html` frame by frame with Playwright and stitches an MP4. First run installs Chromium (~1 min). You can also open `viewer/index.html` locally in any browser and drop `runs/<name>/run3d.json` on it for an interactive orbit/follow view.

In [ ]:
#@title 8b. 3D MP4  { display-mode: "form" }
CAMERA = "orbit"  #@param ["orbit", "top", "follow:A", "follow:B"]
FPS = 30  #@param {type:"number"}
!pip install -q playwright > /dev/null && playwright install chromium > /dev/null 2>&1 && playwright install-deps chromium > /dev/null 2>&1
from flypair.video3d import render3d
mp4_3d = render3d(live, d / "arena3d.mp4", fps=FPS, camera=CAMERA)
b64 = base64.b64encode(open(mp4_3d, "rb").read()).decode()
display(HTML(f'<video width="900" controls src="data:video/mp4;base64,{b64}"></video>'))
files.download(str(mp4_3d))

## 9. Optional: LLM dub (post-hoc captions, never fed back into the sim)
Add `ANTHROPIC_API_KEY` under the 🔑 *Secrets* panel on the left and enable notebook access. Output is watermarked **“LLM dub — not fly output”**.

In [ ]:
#@title 9. Dub  { display-mode: "form" }
DO_DUB = False  #@param {type:"boolean"}
if DO_DUB:
    !pip install -q anthropic
    from google.colab import userdata
    from flypair.dub import dub, dubbed_video
    caps = dub(live, api_key=userdata.get("ANTHROPIC_API_KEY"), window_ms=500)
    out = dubbed_video(live, d / "arena_dubbed.mp4", caps)
    b64 = base64.b64encode(open(out, "rb").read()).decode()
    display(HTML(f'<video width="900" controls src="data:video/mp4;base64,{b64}"></video>'))
    files.download(str(out))